In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages_CNN', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

import numpy as np
import pandas as pd
import itertools
import math

In [ ]:
#@title Read the data frame from csv file
save_dir = './drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/'
filtered_combined_df = pd.read_csv(os.path.join(save_dir, 'all_data_neg_part_4.csv'))

In [ ]:
#@title Extract the negative patch information from the data frame
# randomly pick 500k from total negative patches

import math
import itertools
import numpy as np
import pandas as pd  # Assuming you use pandas for data frame operations

# Create or overwrite cnn_positive_patch_3points.txt at the beginning
with open('./drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/cnn_negative_patch_3points_part4.txt', 'w') as f:
    f.write('')  # Writing an empty string to ensure the file is empty


# Define your functions as before but with slight modifications
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

# NumPy is more efficient for vectorized operations
def calculate_distance(residue1, residue2):
    x1, y1, z1 = residue1['SCx'], residue1['SCy'], residue1['SCz']
    x2, y2, z2 = residue2['SCx'], residue2['SCy'], residue2['SCz']
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

patch_count = 0

# Pre-compute the total number of rows in the data frame for performance
total_rows = filtered_combined_df[filtered_combined_df['Status'] == 'Non-Interacting'].shape[0]

all_negative_patches = []

for name, group in filtered_combined_df.groupby('name'):
    non_interacting_residues = group[(group['Status'] == 'Non-Interacting' ) & (group['SaSaRate'] > 0.1)]
    patches = []

    # Calculate N for the current protein group
    N = int(7000 * len(group) / total_rows)
    patches_found = 0  # Track how many patches we have found for this protein

    # Shuffle the rows for random selection
    shuffled_residues = non_interacting_residues.sample(frac=1).reset_index(drop=True)

    # one residue is not allowed in two different patches
    # used_residues = set()

    for comb in itertools.combinations(shuffled_residues.iterrows(), 3):
        if patches_found >= N:
            break  # We've found N patches for this protein, move to the next one

        index1, row1 = comb[0]
        index2, row2 = comb[1]
        index3, row3 = comb[2]

        # Skip this combination if any residue is already used in another patch
        # if index1 in used_residues or index2 in used_residues or index3 in used_residues:
        #    continue

        d12 = calculate_distance(row1, row2)
        d23 = calculate_distance(row2, row3)
        d31 = calculate_distance(row3, row1)

        if d12 < 8.5 and d23 < 8.5 and d31 < 8.5:
        #:  # Modified condition for SaSaRate

            patch_info = [[row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                          [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                          [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]]
            patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]
            patches.append({'indices': patch_indices, 'info': patch_info})

            # Mark these residues as used
            # used_residues.update([index1, index2, index3])

            patch_count += 1
            patches_found += 1
            print(f"Patches generated: {patches_found} for protein {name} with current N: {N}")


    # Output patches to a text file
    with open('./drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/cnn_negative_patch_3points_part4.txt', 'a') as f:
        f.write(f"Patches for protein {name}:\n")
        for patch in patches:
            f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

    for patch in patches:
        patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
        for idx, aa_info in enumerate(patch['info']):
            aa = aa_info[0]
            ss = aa_info[1]
            sasa = aa_info[2]
            si = aa_info[3]  # Extract the SI value

            aa_encoded = encode_aa(aa)
            ss_encoded = encode_ss(ss)
            sasa_encoded = encode_sasa(sasa)

            combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

            for i in range(20):
                if aa_encoded[i] == 1:
                    patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value
        all_negative_patches.append(patch_matrix)

# Convert all_negative_patches to a NumPy array
all_negative_patches_3points = np.array(all_negative_patches)

# Save the NumPy array
save_dir = './drive/MyDrive/@antibody/antigen_epitope/abdb/data_processed_step1_with_SI/'
np.save(f'{save_dir}cnn_negative_patch_3points_part4.npy', all_negative_patches_3points)

In [ ]:
len(patches)

In [ ]:
patches[2]